# WAV2HYP catalog explorer

Set **`CONFIG_PATH`** (and optional path overrides) in **Setup**, then run **Catalog** (defines `catalog_df`, `CATALOG_T1`/`T2`, histogram). **Catalog** and **Day** use **VolcanoFigure**; **Event** uses **Map** + **CrossSection** + clipboards via `wav2hyp.viz.sthelens_clipboards` (same code as `analysis_local/sthelens_map_and_clipboards.py`).


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Image, display

from wav2hyp.utils.io import NLLOutput
from wav2hyp.viz.sthelens_clipboards import (
    attach_pick_probabilities,
    make_catalog_volcano_figure,
    render_single_event_combined_pil,
    sthelens_paths_from_wav2hyp_config,
)


## Setup

Run the next two cells after editing paths.


In [ ]:
# Path to your WAV2HYP YAML (inventory + output layout).
# Works whether the notebook's working directory is the repo root or notebooks/.
_cwd = Path.cwd()
if (_cwd / "examples/sthelens.yaml").is_file():
    CONFIG_PATH = (_cwd / "examples/sthelens.yaml").resolve()
elif (_cwd.parent / "examples/sthelens.yaml").is_file():
    CONFIG_PATH = (_cwd.parent / "examples/sthelens.yaml").resolve()
else:
    CONFIG_PATH = (_cwd / "examples/sthelens.yaml").resolve()

# Pipeline output root: must contain locations/nll.h5 unless you set LOCATOR_H5 below.
# If None, uses output.base_dir from the YAML (relative to the YAML file's directory).
RUN_BASE_DIR = None

# Folder with optional pre-rendered figures (catalog map PNG, clipboards/). Defaults to RUN_BASE_DIR.
ARTIFACTS_DIR = None

# Set to an explicit nll.h5 path if the catalog file is not under RUN_BASE_DIR/locations/.
LOCATOR_H5 = None

# Pre-rendered catalog map filename inside ARTIFACTS_DIR (optional).
CATALOG_MAP_PNG_NAME = "sthelens_map.png"
CLIPBOARDS_SUBDIR = "clipboards"


In [ ]:
def _resolve(cfg_path: Path, rel_or_abs: str) -> Path:
    p = Path(rel_or_abs)
    return p.resolve() if p.is_absolute() else (cfg_path.parent / p).resolve()


with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)

RUN_BASE_DIR = Path(RUN_BASE_DIR).resolve() if RUN_BASE_DIR is not None else _resolve(
    CONFIG_PATH, _cfg["output"]["base_dir"]
)
ARTIFACTS_DIR = Path(ARTIFACTS_DIR).resolve() if ARTIFACTS_DIR is not None else RUN_BASE_DIR
loc_sub = _cfg["output"].get("locator_dir", "locations")
LOCATOR_H5 = Path(LOCATOR_H5).resolve() if LOCATOR_H5 is not None else (RUN_BASE_DIR / loc_sub / "nll.h5")
INVENTORY_PATH = _resolve(CONFIG_PATH, _cfg["inventory"]["file"])

nll = NLLOutput(str(LOCATOR_H5))
_paths = sthelens_paths_from_wav2hyp_config(_cfg, config_path=CONFIG_PATH, run_base_dir=RUN_BASE_DIR)
paths = replace(_paths, nll_h5=Path(LOCATOR_H5).resolve())

print("CONFIG_PATH     ", CONFIG_PATH)
print("RUN_BASE_DIR    ", RUN_BASE_DIR)
print("ARTIFACTS_DIR   ", ARTIFACTS_DIR)
print("INVENTORY_PATH  ", INVENTORY_PATH)
print("LOCATOR_H5      ", LOCATOR_H5, "exists:", LOCATOR_H5.exists())
print("VolcanoFigure, Map, CrossSection — catalog map uses VolcanoFigure; per-event stacks use Map + CrossSection in wav2hyp.viz.sthelens_clipboards.")
print("Run the Catalog cell to load catalog_df (time-filtered) and figures.")


## Catalog

Set **`CATALOG_T1`** / **`CATALOG_T2`** (same strings you would pass to CLI ``--t1`` / ``--t2``; `None` = full file). Set **`HISTOGRAM_FREQ`** (pandas offset: ``"1D"``, ``"1h"``, …). Then run the code cell: **VolcanoFigure** overview map, histogram, and optional saved PNG.


In [ ]:
# Time window on catalog_table (None = no bound). Strings are passed to NLLOutput / ObsPy UTCDateTime.
CATALOG_T1 = None  # e.g. "2004-09-23T00:00:00"
CATALOG_T2 = None  # e.g. "2004-09-24T00:00:00"

# Histogram bin width: pandas offset alias, e.g. "1D" (daily), "1h", "6h", "7D"
HISTOGRAM_FREQ = "1D"

catalog_df = nll.read_catalog_table(t1=CATALOG_T1, t2=CATALOG_T2)
print(f"catalog_df rows: {len(catalog_df)}")

cat = catalog_df.copy()
cat["origin_time"] = pd.to_datetime(cat["origin_time"], utc=True, errors="coerce")
_freq = HISTOGRAM_FREQ.strip()
if _freq.lower() in ("1d", "d"):
    _freq = "1D"
hist = cat.groupby(pd.Grouper(key="origin_time", freq=_freq)).size()
hist = hist[hist > 0]

fig, axh = plt.subplots(figsize=(8, 3.5))
hist.plot(kind="bar", ax=axh, color="steelblue")
axh.set_title(f"Events per {HISTOGRAM_FREQ} (UTC)")
axh.tick_params(axis="x", rotation=45)
axh.set_xlabel("Time bin start (UTC)")
plt.tight_layout()
plt.show()

_ = make_catalog_volcano_figure(
    _cfg,
    cat,
    inventory_path=paths.inventory_path,
    map_t1=CATALOG_T1,
    map_t2=CATALOG_T2,
    out_png=None,
)
plt.show()

_map = ARTIFACTS_DIR / CATALOG_MAP_PNG_NAME
if _map.is_file():
    display(Image(filename=str(_map)))
else:
    print(f"No optional pre-rendered map PNG at {_map}")


## Day view

Pick a **UTC calendar date** that overlaps your origins.


In [ ]:
VIEW_DATE = "2004-09-23"


In [ ]:
day_start = pd.Timestamp(VIEW_DATE, tz="UTC")
day_end = day_start + pd.Timedelta(days=1)
day_df = catalog_df[
    (pd.to_datetime(catalog_df["origin_time"], utc=True) >= day_start)
    & (pd.to_datetime(catalog_df["origin_time"], utc=True) < day_end)
].copy()

display(day_df)

day_t1 = day_start.strftime("%Y-%m-%dT%H:%M:%S")
day_t2 = day_end.strftime("%Y-%m-%dT%H:%M:%S")
make_catalog_volcano_figure(
    _cfg,
    day_df,
    inventory_path=paths.inventory_path,
    map_t1=day_t1,
    map_t2=day_t2,
    out_png=None,
)
plt.show()


## Event view

Set **`EVENT_ID`** to the full `event_id` from `catalog_table`. The cell below builds the stacked figure (``Map`` + ``CrossSection`` + waveforms + tables) using :func:`wav2hyp.viz.sthelens_clipboards.render_single_event_combined_pil` — same pipeline as ``analysis_local/sthelens_map_and_clipboards.py``. Requires SDS waveform access from your YAML. Falls back to disk images under **`ARTIFACTS_DIR / CLIPBOARDS_SUBDIR`** if rendering returns nothing.


In [ ]:
EVENT_ID = "paste-full-event_id-here"
# If clipboard PNG names do not include the full EVENT_ID, set a substring that appears in the filename (e.g. short hash).
CLIPBOARD_NAME_SUBSTRING = None


In [ ]:
ev_df = catalog_df.loc[catalog_df["event_id"].astype(str) == str(EVENT_ID)]
display(ev_df)

if len(ev_df) == 1:
    arr = nll.read_arrivals(event_ids=[str(EVENT_ID)])
    arr = attach_pick_probabilities(
        catalog_df,
        arr,
        eqt_volpick_h5=paths.eqt_volpick_h5,
        t1=CATALOG_T1,
        t2=CATALOG_T2,
    )
    display(arr)
    pil_img = render_single_event_combined_pil(
        _cfg, paths, catalog_df, arr, event_id=str(EVENT_ID)
    )
    if pil_img is not None:
        display(pil_img)
    else:
        print("render_single_event_combined_pil returned None (no waveforms or missing event).")
else:
    print("No single match — check EVENT_ID against the catalog table.")

_clip = ARTIFACTS_DIR / CLIPBOARDS_SUBDIR
_placeholder = "paste-full-event_id-here"
_needle = CLIPBOARD_NAME_SUBSTRING or (None if EVENT_ID == _placeholder else EVENT_ID)
if _clip.is_dir() and _needle:
    hits = sorted(p for p in _clip.iterdir() if _needle in p.name)
    for p in hits:
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".gif", ".webp"}:
            display(Image(filename=str(p)))
    if not hits:
        print(f"No clipboard files under {_clip} with {_needle!r} in the name.")
elif _clip.is_dir():
    print("Set EVENT_ID (and CLIPBOARD_NAME_SUBSTRING if needed) to search clipboards.")
else:
    print(f"No clipboards directory at {_clip} (optional)")
